# Udemy Dataset Cleaning

This notebook performs data cleaning on the UdemyCleanedTitle.csv dataset. The cleaning steps include:
1. Reading and examining the data
2. Handling missing values
3. Standardizing text formats
4. Processing numeric fields (price, ratings, etc.)
5. Cleaning special characters
6. Handling categorical data
7. Saving the cleaned dataset

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import re

# Read the dataset
df = pd.read_csv('../Data/raw/UdemyCleanedTitle.csv')

In [2]:
# Examine the data
print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum())
print("\nData Types:\n", df.dtypes)
print("\nSample of first few rows:\n", df.head())

Dataset Shape: (3683, 13)

Columns: ['course_id', 'course_title', 'url', 'is_paid', 'price', 'num_subscribers', 'num_reviews', 'num_lectures', 'level', 'content_duration', 'published_timestamp', 'subject', 'Clean_title']

Missing Values:
 course_id               0
course_title            0
url                     0
is_paid                 0
price                   0
num_subscribers         0
num_reviews             0
num_lectures            0
level                   0
content_duration        0
published_timestamp     0
subject                 0
Clean_title            20
dtype: int64

Data Types:
 course_id               int64
course_title           object
url                    object
is_paid                object
price                  object
num_subscribers         int64
num_reviews             int64
num_lectures            int64
level                  object
content_duration       object
published_timestamp    object
subject                object
Clean_title            object
dtype:

In [3]:
# 1. Clean special characters and standardize text
def clean_text(text):
    if pd.isna(text):
        return text
    # Replace special characters and standardize text
    text = str(text)
    text = text.replace('�', "'")  # Replace smart quotes
    text = text.replace('�', '-')  # Replace em dash
    text = text.replace('�', 'e')  # Replace accented e
    text = re.sub(r'\s+', ' ', text)  # Remove multiple spaces
    return text.strip()

# Apply cleaning to text columns
text_columns = ['course_title', 'course_description', 'Requirements', 'what_you_will_learn']
for col in text_columns:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

# 2. Handle missing values
df = df.replace('', np.nan)  # Convert empty strings to NaN

# 3. Process numeric fields
numeric_columns = {
    'price': lambda x: pd.to_numeric(str(x).replace('$', '').replace(',', ''), errors='coerce'),
    'num_subscribers': lambda x: pd.to_numeric(str(x).replace(',', ''), errors='coerce'),
    'num_reviews': lambda x: pd.to_numeric(str(x).replace(',', ''), errors='coerce'),
    'num_lectures': lambda x: pd.to_numeric(str(x).replace(',', ''), errors='coerce'),
    'content_duration': lambda x: pd.to_numeric(str(x).replace(',', ''), errors='coerce')
}

for col, func in numeric_columns.items():
    if col in df.columns:
        df[col] = df[col].apply(func)

# 4. Clean and standardize level column if it exists
if 'level' in df.columns:
    df['level'] = df['level'].str.strip().str.lower()
    level_mapping = {
        'all levels': 'all_levels',
        'beginner level': 'beginner',
        'intermediate level': 'intermediate',
        'expert level': 'expert',
        'beginner': 'beginner',
        'intermediate': 'intermediate',
        'expert': 'expert'
    }
    df['level'] = df['level'].map(level_mapping)

# 5. Process subject/category columns if they exist
if 'subject' in df.columns:
    df['subject'] = df['subject'].str.strip().str.lower()

# 6. Clean URL fields if they exist
if 'url' in df.columns:
    df['url'] = df['url'].str.strip()

# 7. Remove duplicates based on course title and URL if available
if 'url' in df.columns:
    df.drop_duplicates(subset=['course_title', 'url'], inplace=True)
else:
    df.drop_duplicates(subset=['course_title'], inplace=True)

# 8. Reset index
df.reset_index(drop=True, inplace=True)

In [4]:
# Examine the cleaned data
print("Final Dataset Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())
print("\nData Types:\n", df.dtypes)

# Display numeric column statistics
numeric_cols = df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    print("\nNumeric Column Statistics:")
    print(df[numeric_cols].describe())

# Display sample of cleaned data
print("\nSample of cleaned data:\n")
print(df.head())

# Save the cleaned dataset
output_path = '../Data/processed/udemy_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"\nCleaned dataset saved to: {output_path}")

Final Dataset Shape: (3677, 13)

Missing Values:
 course_id                 0
course_title              0
url                       0
is_paid                   0
price                   311
num_subscribers           0
num_reviews               0
num_lectures              0
level                     1
content_duration       3676
published_timestamp       0
subject                   0
Clean_title              20
dtype: int64

Data Types:
 course_id                int64
course_title            object
url                     object
is_paid                 object
price                  float64
num_subscribers          int64
num_reviews              int64
num_lectures             int64
level                   object
content_duration       float64
published_timestamp     object
subject                 object
Clean_title             object
dtype: object

Numeric Column Statistics:
          course_id        price  num_subscribers   num_reviews  num_lectures  \
count  3.677000e+03  3366.000000 

# Data Cleaning Summary

The following cleaning steps were performed on the Udemy dataset:

1. **Special Characters Cleaning**:
   - Replaced special quotes, dashes, and accented characters
   - Removed multiple spaces
   - Standardized text formatting in course titles and descriptions

2. **Missing Values**:
   - Converted empty strings to NaN
   - Handled missing values appropriately for each column type

3. **Numeric Data Processing**:
   - Cleaned and converted price fields (removed $ and commas)
   - Standardized subscriber counts and review numbers
   - Converted content duration to numeric format
   - Processed lecture counts

4. **Text Standardization**:
   - Cleaned course titles and descriptions
   - Standardized requirements and learning objectives
   - Processed URLs for consistency

5. **Categorical Data**:
   - Standardized level categories
   - Cleaned subject/category information
   - Normalized text fields

6. **Data Quality**:
   - Removed duplicate entries based on course title and URL
   - Reset index for clean data structure
   - Validated numeric field conversions

The cleaned dataset has been saved to: '../Data/processed/udemy_cleaned.csv'